# Problem 1 — Musical Influence Network

Prepare `influence_data.csv` and construct a directed artist influence network. An edge `influencer → follower` records an observed influence relationship. The data provide no numerical strength, so the baseline graph is unweighted (`weight=1`). Active-start decades are retained as metadata and summarized, but are not used to remove edges.

In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import eigs, spsolve
from IPython.display import display

DATA_FILE = Path('ProblemC_Music_data/influence_data.csv')
if not DATA_FILE.exists():
    DATA_FILE = Path('../ProblemC_Music_data/influence_data.csv')
if not DATA_FILE.exists():
    raise FileNotFoundError('Run this notebook from the project folder or update DATA_FILE.')
print(f'Using: {DATA_FILE.resolve()}')

## 1. Load and inspect
Artist IDs are identifiers, so load them as strings.

In [ ]:
id_cols = ['influencer_id', 'follower_id']
raw = pd.read_csv(DATA_FILE, dtype={c: 'string' for c in id_cols}, encoding='utf-8')
print(f'Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
display(raw.head())
display(raw.dtypes.to_frame('dtype'))

## 2. Clean and validate the edge list
Trim text and IDs, parse start decades, remove rows without endpoint IDs, and collapse duplicate directed pairs. No row is filtered based on time.

In [ ]:
edges = raw.copy()
text_cols = ['influencer_name', 'influencer_main_genre', 'follower_name', 'follower_main_genre']
for c in text_cols:
    edges[c] = edges[c].astype('string').str.strip()
for c in id_cols:
    edges[c] = edges[c].str.strip()
year_cols = ['influencer_active_start', 'follower_active_start']
for c in year_cols:
    edges[c] = pd.to_numeric(edges[c], errors='coerce').astype('Int64')
missing_before = edges.isna().sum()
bad_id = edges[id_cols].isna().any(axis=1)
edges = edges.loc[~bad_id].copy()
duplicate_pairs = int(edges.duplicated(['influencer_id', 'follower_id']).sum())
edges = edges.drop_duplicates(['influencer_id', 'follower_id']).reset_index(drop=True)
print('Missing values before cleaning:'); display(missing_before.to_frame('count'))
print(f'Removed missing endpoint IDs: {bad_id.sum():,}; duplicate pairs: {duplicate_pairs:,}; retained edges: {len(edges):,}')

## 3. Create one metadata row per artist
Artists may occur in either endpoint role. Metadata conflicts are reported; the first observed non-missing value is used for the node attribute, while edge rows are preserved.

In [ ]:
left = edges[['influencer_id','influencer_name','influencer_main_genre','influencer_active_start']].rename(columns={'influencer_id':'artist_id','influencer_name':'artist_name','influencer_main_genre':'main_genre','influencer_active_start':'active_start'})
right = edges[['follower_id','follower_name','follower_main_genre','follower_active_start']].rename(columns={'follower_id':'artist_id','follower_name':'artist_name','follower_main_genre':'main_genre','follower_active_start':'active_start'})
records = pd.concat([left, right], ignore_index=True)
meta_cols = ['artist_name','main_genre','active_start']
conflicts = records.groupby('artist_id')[meta_cols].nunique(dropna=True).gt(1).sum()
nodes = records.sort_values('artist_id').groupby('artist_id', as_index=False)[meta_cols].agg(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA)
print(f'Unique artists: {len(nodes):,}')
print('Artist IDs with conflicting metadata values:'); display(conflicts.to_frame('count'))
display(nodes.head())

## 4. Build the directed graph
`active_start` remains a node attribute. Edge orientation is taken directly from the source data, and all retained relationships receive unit weight.

In [ ]:
G = nx.DiGraph()
for r in nodes.itertuples(index=False):
    G.add_node(r.artist_id, name=r.artist_name, main_genre=r.main_genre, active_start=int(r.active_start) if pd.notna(r.active_start) else None)
for r in edges.itertuples(index=False):
    G.add_edge(r.influencer_id, r.follower_id, weight=1.0)
print(f'Nodes: {G.number_of_nodes():,}')
print(f'Directed edges: {G.number_of_edges():,}')
print(f'Directed density: {nx.density(G):.6f}')
print(f'Self-loops: {nx.number_of_selfloops(G):,}')
print(f'Weak components: {nx.number_weakly_connected_components(G):,}')
print(f'Strong components: {nx.number_strongly_connected_components(G):,}')

## 5. Describe timing without using it as an edge filter
The decade fields indicate when an artist began an active career, not when a specific influence occurred. We summarize their ordering and retain all reported edges, including cases where the influencer's recorded start decade is later.

In [ ]:
timing = edges.dropna(subset=year_cols).copy()
timing['decade_gap'] = timing['follower_active_start'] - timing['influencer_active_start']
display(pd.Series({'edges with both decades':len(timing),'influencer starts earlier':int((timing.decade_gap>0).sum()),'same decade':int((timing.decade_gap==0).sum()),'influencer starts later':int((timing.decade_gap<0).sum()),'median gap (follower minus influencer)':timing.decade_gap.median()}, name='value').to_frame())
display(timing.loc[timing.decade_gap<0, ['influencer_name','influencer_active_start','follower_name','follower_active_start']].head(10))

## 6. Baseline network indicators
Out-degree measures direct followers. Outgoing Katz centrality sums downstream influence paths with geometric attenuation. PageRank on the reversed graph is a comparison measure whose score flows from followers back to the artists they name. These are network-structure measures, not causal effects.

In [ ]:
artist_ids = sorted(G.nodes())
A = nx.to_scipy_sparse_array(G, nodelist=artist_ids, weight='weight', format='csr', dtype=float)
spectral_radius = float(abs(eigs(A, k=1, which='LM', return_eigenvectors=False)[0]))
alpha = 0.10
if spectral_radius > 0 and alpha >= 1 / spectral_radius:
    raise ValueError('Choose alpha below the Katz convergence bound.')
katz = spsolve(sparse.eye(A.shape[0], format='csc') - alpha * A.tocsc(), np.ones(A.shape[0]))
out_degree = dict(G.out_degree())
in_degree = dict(G.in_degree())
reverse_pr = nx.pagerank(G.reverse(copy=False), alpha=0.85)
indicators = nodes.set_index('artist_id').copy()
indicators['out_degree'] = pd.Series(out_degree)
indicators['in_degree'] = pd.Series(in_degree)
indicators['katz_out'] = pd.Series(dict(zip(artist_ids, katz)))
indicators['pagerank_reverse'] = pd.Series(reverse_pr)
print(f'Katz alpha={alpha}; spectral radius={spectral_radius:.4f}; upper bound={1/spectral_radius:.4f}')
display(indicators.sort_values('katz_out', ascending=False).head(20)[['artist_name','main_genre','active_start','out_degree','katz_out','pagerank_reverse']])

## 7. Visualize a readable subnet
Show 12 high out-degree artists and up to 5 of their followers each (favoring followers with high in-degree). Orange marks selected influencers; blue marks sampled followers. This focused view is exploratory, not a complete rendering of the graph.

In [ ]:
top_ids = indicators.sort_values('out_degree', ascending=False).head(12).index.tolist()
selected_edges = []
for artist_id in top_ids:
    targets = sorted(G.successors(artist_id), key=lambda v: indicators.loc[v, 'in_degree'], reverse=True)[:5]
    selected_edges.extend((artist_id, target) for target in targets)
view = nx.DiGraph()
view.add_edges_from(selected_edges)
pos = nx.spring_layout(view, seed=42, k=0.8, iterations=100)
max_degree = max(indicators.out_degree.max(), 1)
sizes = [100 + 1100 * indicators.loc[v, 'out_degree'] / max_degree for v in view.nodes()]
colors = ['#d95f02' if v in top_ids else '#8da0cb' for v in view.nodes()]
labels = {v:G.nodes[v].get('name',v) for v in view.nodes()}
plt.figure(figsize=(16,11))
nx.draw_networkx_edges(view, pos, alpha=.35, arrows=True, arrowsize=10, width=1.0)
nx.draw_networkx_nodes(view, pos, node_size=sizes, node_color=colors, alpha=.9, linewidths=.5, edgecolors='white')
nx.draw_networkx_labels(view, pos, labels=labels, font_size=8)
plt.title('Selected influence ties around high out-degree artists')
plt.axis('off'); plt.tight_layout(); plt.show()

## 8. Optional export for later questions
Run this cell to save reusable cleaned edge, node, and indicator tables under `ProblemC_Q1_outputs/`.

In [ ]:
output_dir = Path('ProblemC_Q1_outputs')
output_dir.mkdir(exist_ok=True)
edges.to_csv(output_dir / 'clean_influence_edges.csv', index=False)
nodes.to_csv(output_dir / 'artist_nodes.csv', index=False)
indicators.reset_index().to_csv(output_dir / 'artist_network_indicators.csv', index=False)
print(f'Saved outputs in {output_dir.resolve()}')